In [ ]:
# !mkdir src
# !pip install -q sacrebleu evaluate

In [ ]:
%%writefile src/config.py

hf_user_name = "amin-oj"
lang1="en"
lang2="fr"
model_checkpoint = f"Helsinki-NLP/opus-mt-{lang1}-{lang2}"

dataset_dict = dict(
    path = "kde4",
    revision = "refs/convert/parquet",
    data_dir = f"{lang1}-{lang2}"
)
train_size = 1024 * 16
test_size = train_size // 4
push_to_hub=False
gacc_steps = 2
train_bs = 32
eval_bs = 64
num_train_epochs = 2
lr=2e-5
eval_on_start = False
lr_scheduler_type = "linear"
task = "translation"
max_length = 128
ckpt_name = f"{model_checkpoint.split("/")[-1]}-finetuned-{task}-{dataset_dict['path']}"

In [ ]:
%%writefile src/utils.py

from kaggle_secrets import UserSecretsClient
import os, torch

def hf_login():
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

def get_mp():
    bf16_supported = False
    if torch.cuda.is_available():
        try:
            bf16_supported = torch.cuda.is_bf16_supported()
        except Exception:
            bf16_supported = False

    mixed_precision = "bf16" if bf16_supported else "fp16"
    return mixed_precision

In [ ]:
%%writefile src/preprocess.py

from src.config import max_length

def preprocess_function(examples, tokenizer):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    model_inputs = tokenizer(
        inputs, text_target=targets,
        max_length=max_length,
        truncation=True,
        # return_tensors="pt" # raises error with no padding
    )
    return model_inputs

In [ ]:
%%writefile src/evaluation.py

import numpy as np
import torch, evaluate
from tqdm.auto import tqdm
from src.config import max_length

def postprocess(predictions, labels, tokenizer):
    predictions = predictions.cpu().numpy()
    labels = labels.cpu().numpy()

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    return decoded_preds, decoded_labels


def run_eval(model, eval_dataloader, accelerator, tokenizer):
    metric = evaluate.load("sacrebleu")
    model.eval()
    pbar = tqdm(
        eval_dataloader,
        desc="Evaluation",
        disable=not accelerator.is_main_process,
        leave=False
    )
    for batch in pbar:
        with torch.no_grad():
            # we unwrap the model to have access to the `generate` method
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch["input_ids"],
                attention_mask=batch["attention_mask"],
                # to make it faster=========
                max_new_tokens=max_length//2,
                num_beams=1,# keep greedy decoding
                use_cache=True,
            )
        labels = batch["labels"]

        # Necessary to pad predictions and labels for gathering
        # Paddings might be different across processes
        generated_tokens = accelerator.pad_across_processes(
            generated_tokens, dim=1, pad_index=tokenizer.pad_token_id
        )
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)

        predictions_gathered = accelerator.gather(generated_tokens)
        labels_gathered = accelerator.gather(labels)

        decoded_preds, decoded_labels = postprocess(predictions_gathered, labels_gathered, tokenizer)
        metric.add_batch(predictions=decoded_preds, references=decoded_labels)

    results = metric.compute()
    return {"BLEU score": results['score']}

In [ ]:
%%writefile src/train.py

from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader
from src.config import train_bs, eval_bs
from src.preprocess import preprocess_function

def create_dls(split_datasets, tokenizer, model):
    tokenized_datasets = split_datasets.map(
        preprocess_function,
        batched=True,
        fn_kwargs = {"tokenizer":tokenizer},
        remove_columns=split_datasets["train"].column_names,
    )
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
    
    train_dataloader = DataLoader(
        tokenized_datasets["train"],
        collate_fn=data_collator,
        batch_size=train_bs,
        shuffle=True,
    )
    eval_dataloader = DataLoader(
        tokenized_datasets["validation"],
        collate_fn=data_collator,
        batch_size=eval_bs
    )
    return train_dataloader, eval_dataloader

In [ ]:
%%writefile main.py

# Load deps
import numpy as np
import os, torch, evaluate,math
from tqdm.auto import tqdm
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, create_repo
from torch.optim import AdamW
from accelerate import Accelerator
from accelerate.utils import DistributedDataParallelKwargs
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer
from transformers import get_scheduler
from src.config import *
from src.utils import hf_login, get_mp
from src.evaluation import postprocess
from src.train import create_dls
from src.evaluation import run_eval


if __name__ == "__main__":
    # Init accelerator
    ddp_kwargs = DistributedDataParallelKwargs(
        find_unused_parameters=True
    )
    accelerator = Accelerator(
        gradient_accumulation_steps=gacc_steps,
        mixed_precision=get_mp(),
        kwargs_handlers=[ddp_kwargs],
    )
    
    if accelerator.is_main_process:
        accelerator.print(f"checkpoint name: {ckpt_name}")
        accelerator.print(f"# GPUs: {accelerator.num_processes}")
        effective_bs = train_bs * gacc_steps * accelerator.num_processes
        accelerator.print(f"effective_batch_size: {effective_bs}")
    # HF Login and repo initiation
    if accelerator.is_main_process:
        hf_login()
        hf_api = HfApi()
        accelerator.print(f"HF user: {hf_api.whoami()['name']}")
        if push_to_hub:
            # Create the HF repo to push the model to the HF hub
            HF_REPO_ID = f"{hf_user_name}/{ckpt_name}"
            create_repo(repo_id=HF_REPO_ID, exist_ok=True)
            accelerator.print(f"HF repo: {HF_REPO_ID}")

    # Load model and tokenizer
    model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
    model.config.use_cache = False
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    # Load and Prep data
    with accelerator.main_process_first():
        raw_datasets = load_dataset(**dataset_dict)
        split_datasets = raw_datasets["train"].train_test_split(
            train_size=train_size,
            test_size=test_size
        )
        split_datasets["validation"] = split_datasets.pop("test")
        train_dataloader, eval_dataloader = create_dls(split_datasets, tokenizer, model)

    if accelerator.is_main_process:
        train_ds = train_dataloader.dataset
        eval_ds = eval_dataloader.dataset
        accelerator.print(f"final input datasets features: {train_ds.features}")
        accelerator.print(f"train DS length: {len(train_ds)}")
        accelerator.print(f"eval DS length: {len(eval_ds)}")

    # Preparing training components
    optimizer = AdamW(model.parameters(), lr=lr)
    model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
        model, optimizer, train_dataloader, eval_dataloader
    )
    num_update_steps_per_epoch = math.ceil(len(train_dataloader) / gacc_steps)
    num_training_steps = num_train_epochs * num_update_steps_per_epoch
    logging_steps = max(1, num_update_steps_per_epoch // 4)
    eval_steps = num_update_steps_per_epoch # eval with `generate()` is expensive
    
    lr_scheduler = get_scheduler(
        lr_scheduler_type,
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )

    accelerator.wait_for_everyone()
    rank = accelerator.process_index
    is_main = accelerator.is_main_process
    pid = f"[rank={rank} main={is_main}]"
    accelerator.print(f"{pid}|train_len={len(train_dataloader)}|eval_len={len(eval_dataloader)}")
    accelerator.print(f"{pid}|eval_steps={eval_steps}|train_opt_steps={num_training_steps}")
    accelerator.wait_for_everyone()



    # Training loop
    global_step = 0
    running_loss = 0.0
    running_mb = 0
    scheduler_steps = 0
    progress_bar = tqdm(
        total=num_training_steps,
        desc="Optimizer Updates",
        disable=not accelerator.is_main_process,
        leave=False,
    )

    if eval_on_start:
        eval_logs = run_eval(model, eval_dataloader, accelerator, tokenizer)
        if accelerator.is_main_process:
            accelerator.print(f"Step {global_step} eval:", eval_logs)

    for epoch in range(num_train_epochs):
        model.train()
        for step, batch in enumerate(train_dataloader):
            with accelerator.accumulate(model):
                with accelerator.autocast():
                    outputs = model(**batch)
                    loss = outputs.loss
                running_loss += loss.detach().float().item()
                running_mb += 1
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    lr_scheduler.step()
                    scheduler_steps += 1
            
            # We count "global_step" in terms of optimizer updates
            if accelerator.sync_gradients:
                global_step += 1
                progress_bar.update(1)

                # TRAIN logging (main process only)
                if global_step % logging_steps == 0:
                    avg_train_loss = running_loss / max(1, running_mb)
                    lr_val = optimizer.param_groups[0]["lr"]
                    loss_t = torch.tensor([avg_train_loss], device=accelerator.device)
                    lr_t   = torch.tensor([lr_val], device=accelerator.device)
                    loss_all = accelerator.gather(loss_t).detach().cpu().tolist()
                    lr_all   = accelerator.gather(lr_t).detach().cpu().tolist()
                    if accelerator.is_main_process:
                        for r, (l, lr_) in enumerate(zip(loss_all, lr_all)):
                            msg = f"train_loss/rank_{r} = {l} | learning_rate/rank_{r} = {lr_}"
                            accelerator.print(msg)
                    running_loss = 0.0
                    running_mb = 0
                # EVAL logging
                if (global_step % eval_steps == 0) or (global_step == num_training_steps):
                    eval_logs = run_eval(model, eval_dataloader, accelerator, tokenizer)
                    msg = f"Step {global_step} eval:", eval_logs
                    if accelerator.is_main_process: accelerator.print(msg)


    if accelerator.is_main_process:
        accelerator.print("Scheduler steps:", scheduler_steps, "Expected:", num_training_steps)
    accelerator.wait_for_everyone()
    progress_bar.close()
    accelerator.end_training()
    accelerator.free_memory()

In [ ]:
! accelerate launch --num_processes 2 main.py